# 05 - Resultados, comparación y conclusiones

Este notebook reúne las métricas y figuras exportadas por los notebooks 03 y 04. Evalúa cinco grados de artrosis sobre una partición de prueba de 248 radiografías.

> Prototipo académico. No es una herramienta de diagnóstico.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown, Image as NotebookImage

PROJECT_DIR = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'reports').exists() and (p / 'notebooks').exists()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError('No se encontró la raíz del proyecto.')
REPORTS_DIR = PROJECT_DIR / 'reports'
FIGURES_DIR = REPORTS_DIR / 'figures'
with open(REPORTS_DIR / 'baseline_metrics.json', encoding='utf-8') as f:
    baseline = json.load(f)
with open(REPORTS_DIR / 'modelos_avanzados_metrics.json', encoding='utf-8') as f:
    advanced = json.load(f)
print('Test:', baseline['n_test_samples'], '| Características:', baseline['feature_dimension'])

Test: 248 | Características: 320


## 1. Comparación final
Las métricas macro dan el mismo peso a cada grado, algo útil ante el desbalance de clases.

In [2]:
comparison = pd.DataFrame(advanced['comparison_table']).sort_values('F1 Macro', ascending=False).reset_index(drop=True)
comparison.insert(0, 'Posición', comparison.index + 1)
display(comparison.round(4))
comparison.to_csv(REPORTS_DIR / 'comparacion_modelos.csv', index=False)
best = comparison.iloc[0]
display(Markdown(f"**Mayor F1 macro:** {best['Modelo']} ({best['F1 Macro']:.4f}). Parámetros elegidos con validación: SVD k={advanced['svd_best_k']}, SVM C={advanced['svm_best_C']}, XGBoost árboles={advanced['xgb_best_n_estimators']}."))

   Posición                             Modelo  ...  Precision Macro  Recall Macro
0         1              XGBoost (300 Árboles)  ...           0.7617        0.6884
1         2                   SVM (Kernel RBF)  ...           0.6968        0.6618
2         3        Random Forest (200 árboles)  ...           0.6242        0.5773
3         4  Regresión Logística L2 (Baseline)  ...           0.5576        0.5903
4         5      SVD Subespacios (Menor Norma)  ...           0.6052        0.5296

[5 rows x 8 columns]
<IPython.core.display.Markdown object>


## 2. Matrices de confusión
Revisar qué grados confunde cada modelo. Dado que la escala es ordinal, los errores entre grados vecinos merecen una interpretación distinta a los errores extremos.

In [3]:
display(NotebookImage(filename=str(FIGURES_DIR / 'modelos_avanzados_confusion_matrices.png')))
display(NotebookImage(filename=str(FIGURES_DIR / 'baseline_confusion_matrix.png')))

<IPython.core.display.Image object>
<IPython.core.display.Image object>


## 3. Curvas ROC, aprendizaje e importancia de variables
La importancia de XGBoost indica uso predictivo de descriptores, no causalidad ni una explicación clínica. Las curvas de aprendizaje ayudan a detectar sobreajuste comparando entrenamiento y validación cruzada.

In [4]:
for name in ['modelos_avanzados_roc_curves.png', 'modelos_avanzados_learning_curves.png', 'modelos_avanzados_feature_importance.png', 'baseline_roc_curve.png', 'baseline_learning_curve.png']:
    path = FIGURES_DIR / name
    if path.exists():
        display(Markdown('### ' + name.replace('_', ' ').replace('.png', '')))
        display(NotebookImage(filename=str(path)))
    else:
        print('Figura faltante:', path)

<IPython.core.display.Markdown object>
<IPython.core.display.Image object>
<IPython.core.display.Markdown object>
<IPython.core.display.Image object>
<IPython.core.display.Markdown object>
<IPython.core.display.Image object>
<IPython.core.display.Markdown object>
<IPython.core.display.Image object>
<IPython.core.display.Markdown object>
<IPython.core.display.Image object>


## 4. Baseline y conclusiones
La regresión logística es la referencia lineal. Se reporta desde el JSON de resultados para evitar transcribir métricas manualmente.

In [5]:
display(pd.DataFrame([baseline['logistic_final_test']]))
print(f"AUC ROC macro baseline: {baseline['roc_auc_macro']:.4f}")

                               Modelo  Accuracy  ...  Precision Macro  Recall Macro
0  Regresión Logística Final (C=0.01)  0.576613  ...         0.557643      0.590259

[1 rows x 7 columns]
AUC ROC macro baseline: 0.8510


### Conclusiones y limitaciones

- En los resultados guardados, XGBoost obtuvo el mayor F1 macro entre los cuatro modelos; la conclusión se limita a esta partición de prueba.
- El grado leve presenta menor rendimiento que los grados más distinguibles; consultar los reportes por clase y matrices antes de resumir errores.
- La evaluación usa 248 imágenes de un único dataset. Hace falta validación externa para afirmar generalización.
- Las imágenes panorámicas bilaterales se procesan separando ambas mitades y promediando sus descriptores; esto conserva una fila por imagen, pero pierde diferencias entre lados.
- El preprocesamiento actual redimensiona a formato cuadrado y puede alterar la relación de aspecto. Debe describirse como limitación; no afirmar que preserva estrictamente la geometría.
- La importancia de variables no demuestra mecanismos radiológicos causales.
- El sistema es experimental y no debe utilizarse para diagnóstico o tratamiento.

## 5. Archivos para la reportería web
El CSV comparativo se guarda en reports. Las figuras se encuentran en reports/figures; usarlas solo si corresponden a la ejecución más reciente de los notebooks 02, 03 y 04.

In [6]:
for rel in ['reports/comparacion_modelos.csv', 'reports/figures/modelos_avanzados_confusion_matrices.png', 'reports/figures/modelos_avanzados_roc_curves.png', 'reports/figures/modelos_avanzados_learning_curves.png', 'reports/figures/modelos_avanzados_feature_importance.png']:
    path = PROJECT_DIR / rel
    print(('OK      ' if path.exists() else 'FALTA   ') + rel)

OK      reports/comparacion_modelos.csv
OK      reports/figures/modelos_avanzados_confusion_matrices.png
OK      reports/figures/modelos_avanzados_roc_curves.png
OK      reports/figures/modelos_avanzados_learning_curves.png
OK      reports/figures/modelos_avanzados_feature_importance.png
